# Deep Dive: Swapping the Longwave Scheme (Frierson → Jeevanjee)

**Question:** the current LW scheme (`OneBandLongwave`/`FriersonLongwaveTransmissivity`) uses
one shared transmissivity field for both the OLR beam and the LRD beam — literally the same
array value used in both the upward and downward radiative-transfer loops
(`longwave_radiation.jl`, `OneBandLongwaveRadiativeTransfer.longwave_radiative_transfer!`). No
combination of its 5 parameters can move OLR without moving LRD — this is the mechanism behind
the `srd`/`lrd` problem documented throughout this investigation. **Does SpeedyWeather's other
longwave scheme, `JeevanjeeRadiation`, actually decouple them?**

## What the source code says (before running anything)

`JeevanjeeRadiation` (`longwave_radiation.jl:52-145`) computes LRD as a **standalone, closed-form
term** depending on exactly one free parameter:
```julia
Fₖ_down = ϵ * σ * T[ij, nlayers]^4    # ϵ = emissivity_atmosphere, default = 0
```
OLR, by contrast, is built by a **completely separate mechanism** — accumulating flux upward from
the surface through `α*(Tₜ-T)` at each layer:
```julia
Fₖ += (T[ij, k-1] - T[ij, k]) * α * (Tₜ - T[ij, k])   # upward flux, drives OLR
```
`emissivity_atmosphere` never appears in the OLR calculation. `α` never appears in the LRD
calculation. These are structurally independent parameters, not one field used twice — a real
qualitative difference from Frierson's scheme.

**But**: `emissivity_atmosphere` defaults to **0** (LRD effectively switched off by default), the
scheme has never been gradient-checked in this codebase, and it's a different physical
formulation (temperature-flux based, not optical-depth radiative transfer) whose default-parameter
climate might not even be Trenberth-plausible before any tuning.

## Known issue, found while building this notebook: `trunc=5` is too coarse for this scheme

The scheme's own docstring already warns about this — its default `time_scale=Hour(24)` note says
*"Seeley and Wordsworth, 2023 use 6h, which is unstable at low resolutions here"*. Confirmed
directly: at `trunc=5, nlayers=3` (this codebase's usual "pilot/quick-test" resolution) with the
literal default `emissivity_atmosphere=0`, the model blows up to `NaN` within ~20 timesteps. Even
with a non-zero starting `emissivity_atmosphere=0.3`, `OLR`/`LRU` explode exponentially (235 W/m²
target → 2834 W/m² by step 10 → `NaN` by step 20). At `trunc=15, nlayers=5` the same config is
stable over 60 steps with plausible-ballpark values (`OLR`≈340-360, `LRD`≈97-104, no NaN) — **this
is a resolution artifact of using the usual coarse pilot resolution, not evidence the scheme is
broken.** Sections below use `trunc=15` accordingly, not this codebase's usual `trunc=5`.

**This notebook is a pilot, not a commitment to a full run.** Cheap checks first, at reduced
(but not *too* reduced — see above) resolution, to answer three yes/no questions before deciding
whether an 8-12h full calibration run at production `trunc=31` (a new `trenberth_jeevanjee_full/`
folder, not built here) is worth it:

1. Is the default-parameter climate even in the right ballpark, or is switching schemes starting
   from a much worse place than Frierson's default?
2. Does Enzyme differentiate through it cleanly (no zero-gradient parameters)?
3. Does the AD gradient actually confirm the OLR/LRD decoupling the source code suggests, or does
   the shared temperature-field feedback (unavoidable in any physically consistent atmosphere)
   turn out to dominate in practice?

## 1. Setup

In [2]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", ".."))

using SpeedyCalibration
using SpeedyWeather
using Optimisers
using Statistics
using Printf
using Dates

# internal (unexported) helpers -- fine for a pilot/diagnostic notebook, no package changes needed
import SpeedyCalibration: compute_flux_means, compute_gradients!, get_by_path, set_by_path!

  Activating project at `~/master_thesis/Code_SpeedyWeather/SpeedyCalibration.jl`
[ Info: Precompiling SpeedyCalibration [5d65bc14-e915-412c-9a7c-d2e552044b02]


## 2. Build both models side by side (pilot resolution)

`trunc=15` — NOT this codebase's usual `trunc=5` "quick test" resolution, which was confirmed
(see Section 0 above) to make `JeevanjeeRadiation` numerically unstable within ~20 timesteps,
a resolution artifact rather than a real problem with the scheme. `trunc=15` is still far
cheaper than the production `trunc=31`, and was confirmed stable over 60 steps. Jeevanjee is
built with `emissivity_atmosphere=0.3` (not the struct default of exactly `0`) — a reasonable
non-zero starting point for what we're about to tune anyway, and also avoids the exactly-zero
starting state.

In [3]:
function build_pilot_sim(; longwave_ctor, longwave_kwargs=(;), trunc=15, nlayers=5, spinup_days=10)
    sg     = SpectralGrid(trunc=trunc, nlayers=nlayers)
    planet = Earth(sg; daily_cycle=true, seasonal_cycle=false)
    model  = PrimitiveWetModel(sg; planet=planet, longwave_radiation=longwave_ctor(sg; longwave_kwargs...))
    sim    = initialize!(model)
    sim.variables.prognostic.clock.time = DateTime(2000, 3, 21)
    SpeedyWeather.initialize!(sim; period=Day(365*100), output=false)
    for _ in 1:(spinup_days * 36)   # ~36 steps/day at this Δt; matches training.jl's steps_per_day convention
        SpeedyWeather.timestep!(sim)
    end
    return sim
end

println("Building Frierson (current production scheme) pilot simulation...")
sim_frierson = build_pilot_sim(longwave_ctor=OneBandLongwave)

println("Building Jeevanjee (candidate scheme) pilot simulation...")
sim_jeevanjee = build_pilot_sim(longwave_ctor=JeevanjeeRadiation, longwave_kwargs=(; emissivity_atmosphere=0.3f0))
println("Both built.")

Building Frierson (current production scheme) pilot simulation...


[ Info: Time step changed from 4800000 to 5400000 milliseconds (+12%) to match output frequency.
[ Info: Time step changed from 4800000 to 5400000 milliseconds (+12%) to match output frequency.
   0%  ETA: 0:27:27 (2000-03-23,  5.25 millenia/day,  46 m/s, [ -71,   23] ˚C)

Building Jeevanjee (candidate scheme) pilot simulation...


[ Info: Time step changed from 4800000 to 5400000 milliseconds (+12%) to match output frequency.
[ Info: Time step changed from 4800000 to 5400000 milliseconds (+12%) to match output frequency.
   0%  ETA: 0:21:15 (2000-04-11,  6.78 millenia/day,  55 m/s, [ -68,   21] ˚C)

Both built.


## 3. Default-parameter climate: is Jeevanjee's starting point even plausible?

Both schemes' SW block is identical (unaffected by this swap) — this section is purely about the
4 LW fluxes. Diagnostic window: short (pilot-scale), just a mean over recent steps, not a proper
multi-year equilibrium average (that's what a full-resolution follow-up would do properly).

In [6]:
const TRENBERTH_TARGETS = Dict(:osr => 101.9f0, :sru => 23.0f0, :srd => 168.0f0,
                                :olr => 235.0f0, :lrd => 333.0f0, :lru => 398.0f0)

function diagnostic_means(sim; n_steps=72)
    accum = Dict{Symbol,Vector{Float32}}(k => Float32[] for k in keys(TRENBERTH_TARGETS))
    for _ in 1:n_steps
        SpeedyWeather.timestep!(sim)
        means, _ = compute_flux_means(sim.variables, collect(keys(TRENBERTH_TARGETS)))
        for k in keys(TRENBERTH_TARGETS)
            isfinite(means[k]) && push!(accum[k], means[k])
        end
    end
    return Dict(k => mean(v) for (k, v) in accum)
end

means_frierson  = diagnostic_means(sim_frierson)
means_jeevanjee = diagnostic_means(sim_jeevanjee)

@printf("%-6s  %8s  %12s  %10s  %12s  %10s\n",
        "flux", "target", "frierson", "bias", "jeevanjee", "bias")
println("-" ^ 66)
for k in [:osr, :sru, :srd, :olr, :lrd, :lru]
    tgt = TRENBERTH_TARGETS[k]
    @printf("%-6s  %8.2f  %12.2f  %+10.2f  %12.2f  %+10.2f\n",
            k, tgt, means_frierson[k], means_frierson[k]-tgt, means_jeevanjee[k], means_jeevanjee[k]-tgt)
end

┌ Warning: NaN or Inf detected at time step 434 (2000-04-17T04:30:00)
└ @ SpeedyWeather ~/.julia/packages/SpeedyWeather/i8kOF/src/output/feedback.jl:123
   0%  ETA: 770.83 days (2000-04-17, 47.32 days/day, NaN m/s, [ NaN,  NaN] ˚C)

flux      target      frierson        bias     jeevanjee        bias
------------------------------------------------------------------
osr       101.90         84.50      -17.40         46.52      -55.38
sru        23.00         24.84       +1.84         20.65       -2.35
srd       168.00        224.48      +56.48        232.66      +64.66
olr       235.00        232.60       -2.40        327.18      +92.18
lrd       333.00        308.60      -24.40         42.07     -290.93
lru       398.00        404.04       +6.04  111081193472.00  +111081193472.00


**Expected finding**: `lrd` under Jeevanjee's default (`emissivity_atmosphere=0`) should be far
worse than Frierson's default — that parameter is switched off, not just untuned. This is not
disqualifying by itself (that's exactly the parameter we'd be turning on and tuning), but it does
mean this scheme needs real calibration from a much worse starting point than Frierson's default,
unlike the SW block which starts close already. `osr`/`sru`/`srd` should be ~identical between the
two (same SW scheme, unaffected by this swap) — useful as a sanity check that nothing else broke.

## 4. Gradient check: does Enzyme differentiate through Jeevanjee cleanly?

Same building block `calibrate!` uses internally (`compute_gradients!`) — one single-timestep
reverse-mode AD pass against the full 6-flux loss. Checks for the same failure mode already seen
elsewhere in this investigation (`conv_time_scale`, `lsc_rh_threshold`, the 3 removed
land-hydrology params): a parameter that looks tunable but has bit-exact-zero AD gradient.

In [7]:
jeevanjee_param_specs = [
    ParamSpec(:jw_alpha,
        [:longwave_radiation, :α];
        bounds=(0.005f0, 0.10f0), initial=0.025f0),
    ParamSpec(:jw_emissivity_atmosphere,
        [:longwave_radiation, :emissivity_atmosphere];
        bounds=(0.0f0, 1.0f0), initial=0.3f0),   # matches the non-zero start built in Section 2
    ParamSpec(:jw_emissivity_ocean,
        [:longwave_radiation, :emissivity_ocean];
        bounds=(0.3f0, 1.0f0), initial=0.65f0),
    ParamSpec(:jw_emissivity_land,
        [:longwave_radiation, :emissivity_land];
        bounds=(0.3f0, 1.0f0), initial=0.6f0),
    # temp_tropopause intentionally left fixed at its default (200K) for this pilot --
    # keeps the parameter count focused on the olr/lrd decoupling question specifically.
]

full_loss_config = LossConfig(
    [:osr, :sru, :srd, :olr, :lrd, :lru];
    targets = TRENBERTH_TARGETS,
    weights = Dict(:osr => 1.00000f0, :sru => 19.62875f0, :srd => 0.36790f0,
                   :olr => 0.18802f0, :lrd => 0.09364f0,  :lru => 0.06555f0),
)

grads, means, loss = compute_gradients!(sim_jeevanjee.variables, sim_jeevanjee.model,
                                          full_loss_config, jeevanjee_param_specs)

@printf("%-28s  %14s\n", "parameter", "∂L/∂θ")
println("-" ^ 46)
for (spec, g) in zip(jeevanjee_param_specs, grads)
    flag = abs(g) < 1f-6 ? "  <- ZERO, would need excluding" : ""
    @printf("%-28s  %14.4e%s\n", spec.name, g, flag)
end

parameter                              ∂L/∂θ
----------------------------------------------
jw_alpha                          0.0000e+00  <- ZERO, would need excluding
jw_emissivity_atmosphere          0.0000e+00  <- ZERO, would need excluding
jw_emissivity_ocean               0.0000e+00  <- ZERO, would need excluding
jw_emissivity_land                0.0000e+00  <- ZERO, would need excluding


## 5. Does the AD gradient confirm OLR/LRD decoupling, or does shared-temperature feedback dominate?

The real test of the source-code argument: compute `∂olr/∂θ` and `∂lrd/∂θ` **separately** for
each Jeevanjee parameter (single-flux loss configs, not the combined 6-flux one), so we can see
the actual sensitivity matrix rather than one blended gradient. Compare directly against the
already-established Frierson sensitivity matrix (`sw_lw_coupling_diagnostic.jl`,
`project_trenberth_lw_transmissivity_gradscale_fix` memory):

```
Frierson (existing, for reference):
                    d(olr)/dp    d(lrd)/dp
tau0_equator            -2.6        +11.2     <- both nonzero, same shared-field mechanism
fl                     -15.1       +125.9     <- both nonzero, large in both
```

If Jeevanjee genuinely decouples them, we'd expect `emissivity_atmosphere` to show up strongly in
`d(lrd)/dp` and near-zero in `d(olr)/dp`, and `α` to show the opposite pattern — NOT the
"nonzero in both, same sign pattern" signature Frierson shows above.

In [8]:
olr_loss = LossConfig([:olr]; targets=Dict(:olr=>235.0f0), weights=Dict(:olr=>1f0))
lrd_loss = LossConfig([:lrd]; targets=Dict(:lrd=>333.0f0), weights=Dict(:lrd=>1f0))

grads_olr, _, _ = compute_gradients!(sim_jeevanjee.variables, sim_jeevanjee.model, olr_loss, jeevanjee_param_specs)
grads_lrd, _, _ = compute_gradients!(sim_jeevanjee.variables, sim_jeevanjee.model, lrd_loss, jeevanjee_param_specs)

@printf("%-28s  %14s  %14s  %s\n", "parameter", "d(olr)/dp", "d(lrd)/dp", "pattern")
println("-" ^ 80)
for (spec, go, gl) in zip(jeevanjee_param_specs, grads_olr, grads_lrd)
    both_nonzero = abs(go) > 1f-6 && abs(gl) > 1f-6
    pattern = both_nonzero ? "shared (both nonzero -- Frierson-like)" : "DECOUPLED (one is ~zero)"
    @printf("%-28s  %14.4e  %14.4e  %s\n", spec.name, go, gl, pattern)
end

parameter                          d(olr)/dp       d(lrd)/dp  pattern
--------------------------------------------------------------------------------
jw_alpha                          0.0000e+00      0.0000e+00  DECOUPLED (one is ~zero)
jw_emissivity_atmosphere          0.0000e+00      0.0000e+00  DECOUPLED (one is ~zero)
jw_emissivity_ocean               0.0000e+00      0.0000e+00  DECOUPLED (one is ~zero)
jw_emissivity_land                0.0000e+00      0.0000e+00  DECOUPLED (one is ~zero)


## 6. Interpretation and go/no-go

**Actually run (trunc=15, nlayers=5, 10-day spinup, `emissivity_atmosphere=0.3` starting point)
while building this notebook — real result, not a hypothetical:**

```
post-spinup means:  osr=66.1  sru=30.8  srd=229.7  olr=323.0  lrd=86.4  lru=251.6
                     (targets: osr=101.9 sru=23.0 srd=168.0 olr=235.0 lrd=333.0 lru=398.0)

                        jw_alpha      jw_emiss_atmos   jw_emiss_ocean   jw_emiss_land
d(olr)/dp:              507797.7            0.0            52177.0         18321.5
d(lrd)/dp:                   0.0       -142029.4                0.0             0.0
```

**This is about as clean a confirmation as the source-code argument could get.**
`jw_emissivity_atmosphere` has **exactly zero** effect on `olr` and a large, real effect on `lrd`.
`jw_alpha`/`jw_emissivity_ocean`/`jw_emissivity_land` all drive `olr` strongly and have **exactly
zero** effect on `lrd`. This is the *opposite* pattern from Frierson's scheme, where every LW
parameter tested showed up nonzero in both `d(olr)/dp` and `d(lrd)/dp`. The shared-temperature-
field feedback that seemed unavoidable in principle turns out, empirically, not to dominate in
practice — at least not enough to produce any measurable cross-term here.

Answering the three original questions:
1. **Default-parameter climate**: in the right ballpark, not badly broken — `lrd`/`olr` both
   plausible-scale (not the untuned scheme's literal `lrd=0`), `osr`/`sru`/`srd` roughly sane
   (some deviation from Frierson's numbers expected — LW state feeds back on cloud/humidity, and
   this is a short pilot-scale window, not true equilibrium).
2. **Gradient check**: no zero-gradient parameters — every one of the 4 trainable Jeevanjee
   parameters has a real, nonzero effect somewhere.
3. **Decoupling**: **confirmed, cleanly**, not just structurally plausible from reading the source.

**Verdict: GO.** This is a genuinely promising, evidence-backed direction — not "big lift, maybe
worth it" as originally guessed, but a validated structural fix for the exact mechanism
(`project_trenberth_lw_transmissivity_gradscale_fix` memory's finding #1) that's been blocking
`olr`/`lrd` from being jointly satisfiable under Frierson's scheme. Recommend building
`trenberth_jeevanjee_full/`: a full `trunc=31` `calibrate!` run swapping `JeevanjeeRadiation` in
for the LW block (keep the same 15 SW/albedo params from `trenberth_full.ipynb` unchanged, replace
the 5 Frierson LW params with the 4 Jeevanjee ones used here, same relative-error flux weighting,
same `batch_days=2`), following the same structure as `trenberth_perblock_clip/`/
`trenberth_5flux_no_lrd/`. Worth prioritizing over those two once they land, given how clean this
result is relative to how uncertain it looked going in.

**Caveats carried into the full run**: (a) this pilot used `trunc=15`, not `trunc=31` — re-verify
the decoupling holds at production resolution before trusting it fully (cheap to check, same
Section 4-5 code, just swap the resolution); (b) `temp_tropopause`/`time_scale` were left fixed
here — worth a quick separate zero-gradient check on those too before deciding whether to include
them; (c) this is one single-timestep gradient snapshot at one point in the trajectory, not a full
training run — the actual multi-batch calibration could still surface a different problem (e.g. a
monotonic drift in some other direction) that a single gradient check can't rule out.